In [5]:
import os, json, httpx, asyncpg, asyncio, joblib
import numpy as np
import pandas as pd
from typing import Annotated, List
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, END
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, ToolMessage, SystemMessage
from langchain_core.tools import tool
from langchain_groq import ChatGroq
from dotenv import load_dotenv

load_dotenv()

# ══════════════════════════════════════════
#  CONFIG
# ══════════════════════════════════════════
SESSION_FARMER_ID = "c59f6f44-1a98-4eaa-8cf0-3581316a32bb"
MANDI_JSON_PATH   = r"D:\CA_content\Python\KissanSathi\krishisarthi-api\mandi_master.json"
MODEL_DIR         = r"D:\CA_content\Python\KissanSathi\krishisarthi-api\Encoder_and_model"
AGMARKNET_URL     = "https://api.data.gov.in/resource/9ef84268-d588-465a-a308-a864a43d0070"
OPEN_METEO_URL    = "https://api.open-meteo.com/v1/forecast"
DB_URL            = os.getenv("DATABASE_URL")

# ── Load ML models once ──
_xgb   = joblib.load(os.path.join(MODEL_DIR, "krishi_twin_xgb_model.pkl"))
_le_crop  = joblib.load(os.path.join(MODEL_DIR, "crop_encoder.pkl"))
_le_state = joblib.load(os.path.join(MODEL_DIR, "state_encoder.pkl"))
print("✅ XGBoost model + encoders loaded")

BENCHMARK_YIELDS = {
    "RICE YIELD (Kg per ha)": 1872.1,
    "WHEAT.YIELD.Kg.per.ha.": 2087.2,
    "MAIZE.YIELD.Kg.per.ha.": 1880.7,
    "SUGARCANE YIELD (Kg per ha)": 5619.3,
    "COTTON.YIELD.Kg.per.ha.": 296.7,
    "PEARL MILLET YIELD (Kg per ha)": 1001.1,
    "CHICKPEA YIELD (Kg per ha)": 817.6,
    "GROUNDNUT YIELD (Kg per ha)": 1152.9,
}

# ── Debug counter (resets each turn) ──
_DBG = {"hop": 0, "tools": {}}

# ══════════════════════════════════════════
#  SYSTEM PROMPT
# ══════════════════════════════════════════
SYSTEM_INSTRUCTIONS = f"""You are KisanSaathi, a helpful agricultural assistant for Indian farmers.
Farmer ID: {SESSION_FARMER_ID}

You have EXACTLY FOUR tools:
  • list_mandis(crop)                  — Lists APMCs in farmer's district for a crop.
  • fetch_crop_price(mandi_name, crop) — Fetches live mandi price.
  • get_weather(days)                  — Fetches weather forecast (days=1/2/3/7).
  • get_crop_advice()                  — Runs XGBoost What-If analysis; returns personalised advice to improve yield and health score.

══ PRICE FLOW ══
STEP 1: No crop mentioned → ask for crop. Do NOT call any tool.
STEP 2: Crop known, no mandi → call list_mandis(crop) ONCE. Show list. Wait.
STEP 3: Mandi selected → call fetch_crop_price ONCE. Show result. STOP.

══ WEATHER FLOW ══
Farmer asks mausam/barish/garmi → call get_weather(days=N) ONCE. STOP.

══ CROP ADVICE FLOW ══
Farmer asks score badhao/loan/yield improve/kya karu → call get_crop_advice() ONCE.
Explain results in simple Hinglish. Tell EXACTLY what to change and WHY. STOP.

══ HARD RULES ══
✗ NEVER call any tool more than once per user turn.
✗ If tool returned data → STOP, show result, do NOT call again.
✓ Respond in Hinglish (Hindi + English mix)."""

# ══════════════════════════════════════════
#  TOOL 1 — list_mandis
# ══════════════════════════════════════════
@tool
async def list_mandis(crop: str) -> str:
    """
    Returns APMC mandis in farmer's district for a given crop.
    Call ONLY when crop name is known but no mandi selected.
    """
    print(f"\n📋 [TOOL] list_mandis(crop='{crop}')")
    try:
        conn = await asyncpg.connect(DB_URL)
        row  = await conn.fetchrow(
            "SELECT state_name, dist_name FROM farmers WHERE id = $1",
            SESSION_FARMER_ID
        )
        await conn.close()
    except Exception as e:
        result = f"❌ DB Error: {e}"
        print(f"   └─ {result}")
        return result

    if not row:
        return "❌ Farmer not found in DB."

    state, district = row["state_name"], row["dist_name"]
    print(f"   └─ Farmer location: {district}, {state}")

    try:
        with open(MANDI_JSON_PATH, "r", encoding="utf-8") as f:
            mandi_data = json.load(f)
    except FileNotFoundError:
        return "❌ Mandi master JSON not found."

    mandis = []
    for s_key, districts in mandi_data.items():
        if s_key.lower() == state.lower():
            for d_key, m_list in districts.items():
                if d_key.lower() == district.lower():
                    mandis = m_list
                    break

    if not mandis:
        return f"⚠️ {district} ({state}) ke liye koi mandi data nahi hai."

    mandi_list = "\n".join(f"  {i+1}. {m}" for i, m in enumerate(mandis))
    result = (
        f"📍 {district}, {state} ke APMCs ({crop} ke liye):\n"
        f"{mandi_list}\n\n"
        "Kaunsi mandi ka bhav chahiye? (Number ya naam batayein)"
    )
    print(f"   └─ Found {len(mandis)} mandis")
    return result


# ══════════════════════════════════════════
#  TOOL 2 — fetch_crop_price
# ══════════════════════════════════════════
@tool
async def fetch_crop_price(mandi_name: str, crop: str) -> str:
    """
    Fetches live price for a crop at a specific APMC mandi from Agmarknet.
    Call ONLY after farmer has selected a mandi from the list.
    """
    print(f"\n💰 [TOOL] fetch_crop_price(mandi='{mandi_name}', crop='{crop}')")

    try:
        conn = await asyncpg.connect(DB_URL)
        row  = await conn.fetchrow(
            "SELECT state_name FROM farmers WHERE id = $1", SESSION_FARMER_ID
        )
        await conn.close()
    except Exception as e:
        return f"❌ DB Error: {e}"

    if not row:
        return "❌ Farmer not found."

    state      = row["state_name"]
    clean_name = mandi_name.replace("APMC", "").strip()
    params = {
        "api-key":            os.getenv("AGMARKNET_API_KEY"),
        "format":             "json",
        "filters[state]":     state,
        "filters[market]":    clean_name,
        "filters[commodity]": crop,
    }
    print(f"   └─ Calling Agmarknet: state={state}, market={clean_name}, commodity={crop}")

    try:
        async with httpx.AsyncClient() as client:
            resp = await client.get(AGMARKNET_URL, params=params, timeout=15.0)
    except httpx.TimeoutException:
        return "⏱️ Government server ne respond nahi kiya. Thodi der baad try karein."

    if resp.status_code != 200:
        print(f"   └─ API Error: HTTP {resp.status_code}")
        return f"❌ API Error {resp.status_code}."

    records = resp.json().get("records", [])
    print(f"   └─ API returned {len(records)} records")

    if not records:
        return (
            f"⚠️ Aaj {crop} ka data {clean_name} mandi mein available nahi hai.\n"
            "Kal try karein ya doosri mandi chunein."
        )
    r = records[0]
    result = (
        f"✅ Live Price\n"
        f"━━━━━━━━━━━━━━━━━━━━━━━\n"
        f"🌾 Fasal  : {r['commodity']}\n"
        f"🏪 Mandi  : {r['market']}\n"
        f"💰 Price  : ₹{r['modal_price']} / Quintal\n"
        f"📅 Date   : {r['arrival_date']}\n"
        f"━━━━━━━━━━━━━━━━━━━━━━━"
    )
    print(f"   └─ Price: ₹{r['modal_price']}/Quintal")
    return result


# ══════════════════════════════════════════
#  TOOL 3 — get_weather
# ══════════════════════════════════════════
@tool
async def get_weather(days: int = 3) -> str:
    """
    Fetches weather forecast for farmer's registered field GPS location.
    Call for mausam/barish/garmi/temperature/forecast questions.
    days: 1=aaj, 2=kal tak, 3=teen din, 7=hafte bhar (default=3).
    """
    print(f"\n🌤️ [TOOL] get_weather(days={days})")
    days = max(1, min(days, 16))

    try:
        conn = await asyncpg.connect(DB_URL)
        row  = await conn.fetchrow(
            "SELECT field_name, city_name, center_lat, center_lon "
            "FROM farm_fields WHERE farmer_id = $1 LIMIT 1",
            SESSION_FARMER_ID
        )
        await conn.close()
    except Exception as e:
        return f"❌ DB Error: {e}"

    if not row:
        return "⚠️ Khet registered nahi hai. Pehle khet register karein."

    lat = float(row["center_lat"])
    lon = float(row["center_lon"])
    loc = f"{row['field_name'] or 'Khet'}, {row['city_name'] or ''}"
    print(f"   └─ GPS: ({lat:.4f}, {lon:.4f}) | Location: {loc}")

    params = {
        "latitude": lat, "longitude": lon,
        "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum,windspeed_10m_max",
        "forecast_days": days, "timezone": "Asia/Kolkata",
    }
    try:
        async with httpx.AsyncClient() as client:
            resp = await client.get(OPEN_METEO_URL, params=params, timeout=15.0)
        resp.raise_for_status()
    except Exception as e:
        return f"⚠️ Weather API error: {e}"

    daily = resp.json().get("daily", {})
    dates = daily.get("time", [])
    t_max = daily.get("temperature_2m_max", [])
    t_min = daily.get("temperature_2m_min", [])
    rain  = daily.get("precipitation_sum", [])
    wind  = daily.get("windspeed_10m_max", [])

    if not dates:
        return "⚠️ Mausam data available nahi hai."

    lines = [f"🌤️  Mausam — {loc}", f"📍 GPS: ({lat:.4f}, {lon:.4f})", "━" * 35]
    for i in range(min(days, len(dates))):
        r_mm = rain[i] or 0.0
        w_kmh = wind[i] or 0.0
        rain_txt = (f"🌧️  {r_mm}mm (Kaam band!)" if r_mm >= 10
                    else f"🌦️  {r_mm}mm (Halki)" if r_mm > 0 else "☀️  Baarish nahi")
        lines.append(
            f"\n📅 {dates[i]}\n"
            f"   🌡️  {t_min[i]}°C – {t_max[i]}°C\n"
            f"   {rain_txt}\n"
            f"   💨 {w_kmh} km/h{'  ⚠️ Tej aandhi!' if w_kmh > 40 else ''}"
        )
    lines.append("\n" + "━" * 35)
    print(f"   └─ Fetched {len(dates)} days of forecast")
    return "\n".join(lines)


# ══════════════════════════════════════════
#  TOOL 4 — get_crop_advice (What-If Engine)
# ══════════════════════════════════════════

def _compute_health_score(predicted_yield, soil_health_score, npk, wdi,
                           irr_ratio, kharif_rain, kharif_temp, rabi_temp,
                           ndvi, crop_type) -> float:
    benchmark = BENCHMARK_YIELDS.get(crop_type, 2000.0)
    yield_sc  = min((predicted_yield / benchmark) * 100, 100) if benchmark > 0 else 50.0
    soil_base = min((soil_health_score / 200) * 100, 100)
    npk_pen   = min((abs(npk - 120) / 120) * 30, 30)
    soil_sc   = max(soil_base - npk_pen, 0)
    wdi_sc    = (1 - wdi) * 100
    irr_sc    = max((1 - abs(1 - irr_ratio)) * 100, 0)
    rain_sc   = min((kharif_rain / 600) * 100, 100)
    water_sc  = 0.5 * wdi_sc + 0.3 * irr_sc + 0.2 * rain_sc
    clim_sc   = max(100 - max(0, kharif_temp - 35) * 5 - max(0, rabi_temp - 25) * 5, 0)
    ndvi_sc   = min(ndvi * 125, 100) if ndvi is not None else min(yield_sc * 0.8, 100)
    return round(yield_sc*0.25 + soil_sc*0.20 + water_sc*0.25 + clim_sc*0.15 + ndvi_sc*0.15, 2)


def _predict_yield(inputs: dict) -> float:
    try:
        crop_enc  = int(_le_crop.transform([inputs["Crop_Type"]])[0])
    except ValueError:
        crop_enc  = len(_le_crop.classes_) // 2
    try:
        state_enc = int(_le_state.transform([inputs["State_Name"]])[0])
    except ValueError:
        state_enc = len(_le_state.classes_) // 2

    feat_df = pd.DataFrame([{
        "year":                       inputs.get("year", 2026),
        "State_Encoded":              state_enc,
        "Crop_Encoded":               crop_enc,
        "NPK_Intensity_KgHa":         inputs["NPK_Intensity_KgHa"],
        "Irrigation_Intensity_Ratio": inputs["Irrigation_Intensity_Ratio"],
        "WDI":                        inputs["WDI"],
        "Kharif_Avg_MaxTemp":         inputs["Kharif_Avg_MaxTemp"],
        "Kharif_Total_Rain":          inputs["Kharif_Total_Rain"],
        "Rabi_Avg_MaxTemp":           inputs["Rabi_Avg_MaxTemp"],
        "District_Soil_Health_Score": inputs["District_Soil_Health_Score"],
    }])
    log_pred = float(_xgb.predict(feat_df)[0])
    return float(np.expm1(min(max(log_pred, 0.0), 11.0)))


@tool
async def get_crop_advice() -> str:
    """
    Runs XGBoost What-If analysis on the farmer's current inputs.
    Varies NPK (60-160 kg/ha) and Irrigation ratio (0.3-1.0) across
    11 scenarios to find which changes improve yield and health score most.
    Call when farmer asks: score badhao, loan nahi mila, yield kaise badhega,
    kya karu, koi advice do, score improve karna hai.
    """
    print(f"\n🌱 [TOOL] get_crop_advice() — Starting What-If analysis")

    # ── Step 1: Fetch farmer inputs from DB ────────────────
    try:
        conn = await asyncpg.connect(DB_URL)
        farm = await conn.fetchrow(
            """
            SELECT ff.id AS field_id, ff.city_name, ff.state_name,
                   f.state_name AS farmer_state, f.dist_name AS farmer_dist
            FROM   farm_fields ff
            JOIN   farmers f ON f.id = ff.farmer_id
            WHERE  ff.farmer_id = $1
            ORDER  BY ff.created_at DESC LIMIT 1
            """,
            SESSION_FARMER_ID
        )
        if not farm:
            await conn.close()
            return "⚠️ Khet registered nahi hai. Pehle khet register karein."

        print(f"   └─ Farm found: {farm['city_name']}, {farm['state_name']}")

        season = await conn.fetchrow(
            """
            SELECT crop_type, npk_input, irrigation_ratio, wdi_used,
                   ndvi_value, final_health_score, predicted_yield, year,
                   kharif_temp_used, kharif_rain_used, rabi_temp_used, soil_score_used
            FROM   field_predictions
            WHERE  field_id = $1
            ORDER  BY year DESC, calculated_at DESC LIMIT 1
            """,
            str(farm["field_id"])
        )
        if not season:
            await conn.close()
            return "⚠️ Koi prediction data nahi hai. Pehle POST /predict call karein."

        print(f"   └─ Season data: crop={season['crop_type']}, score={season['final_health_score']:.1f}")

        dist_name = (farm["city_name"] or farm["farmer_dist"] or "").lower().strip()
        climate = await conn.fetchrow(
            """
            SELECT AVG(kharif_avg_maxtemp) AS kt, AVG(kharif_total_rain) AS kr,
                   AVG(rabi_avg_maxtemp)   AS rt, AVG(district_soil_health_score) AS ss
            FROM   district_climate_history
            WHERE  LOWER(dist_name) = $1
            """,
            dist_name
        )
        await conn.close()

    except Exception as e:
        return f"❌ DB Error: {e}"

    # ── Step 2: Build base inputs ───────────────────────────
    def _v(val, fallback):
        return float(val) if val is not None else fallback

    c = climate or {}
    base = {
        "State_Name":                 farm["state_name"] or farm["farmer_state"],
        "Crop_Type":                  season["crop_type"],
        "year":                       season["year"] or 2026,
        "NPK_Intensity_KgHa":         _v(season["npk_input"],        120.0),
        "Irrigation_Intensity_Ratio": _v(season["irrigation_ratio"],  0.5),
        "WDI":                        _v(season["wdi_used"],          _v(c.get("wdi"), 0.5)),
        "Kharif_Avg_MaxTemp":         _v(season["kharif_temp_used"],  _v(c.get("kt"), 32.0)),
        "Kharif_Total_Rain":          _v(season["kharif_rain_used"],  _v(c.get("kr"), 900.0)),
        "Rabi_Avg_MaxTemp":           _v(season["rabi_temp_used"],    _v(c.get("rt"), 26.0)),
        "District_Soil_Health_Score": _v(season["soil_score_used"],   _v(c.get("ss"), 50.0)),
        "ndvi":                       season["ndvi_value"],
    }

    current_score = _v(season["final_health_score"], 50.0)
    current_yield = _v(season["predicted_yield"], 0.0)
    orig_npk = base["NPK_Intensity_KgHa"]
    orig_irr = base["Irrigation_Intensity_Ratio"]

    print(f"   └─ Base: NPK={orig_npk:.0f} kg/ha | IRR={orig_irr:.2f} | Score={current_score:.1f}")

    # ── Step 3: Run What-If scenarios ──────────────────────
    suggestions = []

    print(f"   └─ Running NPK scenarios: [60,80,100,120,140,160] kg/ha")
    for npk_val in [60, 80, 100, 120, 140, 160]:
        if abs(npk_val - orig_npk) < 5:
            continue
        m = {**base, "NPK_Intensity_KgHa": float(npk_val)}
        try:
            new_yield = _predict_yield(m)
            new_score = _compute_health_score(
                new_yield, m["District_Soil_Health_Score"], npk_val,
                m["WDI"], m["Irrigation_Intensity_Ratio"],
                m["Kharif_Total_Rain"], m["Kharif_Avg_MaxTemp"],
                m["Rabi_Avg_MaxTemp"], m["ndvi"], m["Crop_Type"]
            )
            gain = new_score - current_score
            print(f"     NPK={npk_val}: yield={new_yield:.0f} kg/ha | score={new_score:.1f} | gain={gain:+.1f}")
            if gain > 0.5:
                suggestions.append({
                    "change":        f"NPK {orig_npk:.0f} → {npk_val} kg/ha",
                    "category":      "fertilizer",
                    "new_yield":     round(new_yield, 0),
                    "new_score":     round(new_score, 1),
                    "score_gain":    round(gain, 1),
                    "action":        f"Urvarak (fertilizer) ko {npk_val} kg/ha kar do.",
                    "why":           "Adhik NPK soil quality ko kharab karta hai aur score girata hai."
                    if npk_val < orig_npk else
                    "Thoda zyada NPK crop nutrition improve karta hai."
                })
        except Exception as e:
            print(f"     ⚠️  NPK={npk_val} scenario failed: {e}")

    print(f"   └─ Running Irrigation scenarios: [0.3,0.5,0.7,0.85,1.0]")
    for irr_val in [0.3, 0.5, 0.7, 0.85, 1.0]:
        if abs(irr_val - orig_irr) < 0.05:
            continue
        m = {**base, "Irrigation_Intensity_Ratio": float(irr_val)}
        try:
            new_yield = _predict_yield(m)
            new_score = _compute_health_score(
                new_yield, m["District_Soil_Health_Score"], m["NPK_Intensity_KgHa"],
                m["WDI"], irr_val,
                m["Kharif_Total_Rain"], m["Kharif_Avg_MaxTemp"],
                m["Rabi_Avg_MaxTemp"], m["ndvi"], m["Crop_Type"]
            )
            gain = new_score - current_score
            print(f"     IRR={irr_val:.2f}: yield={new_yield:.0f} kg/ha | score={new_score:.1f} | gain={gain:+.1f}")
            if gain > 0.5:
                suggestions.append({
                    "change":     f"Irrigation {orig_irr:.0%} → {irr_val:.0%}",
                    "category":   "irrigation",
                    "new_yield":  round(new_yield, 0),
                    "new_score":  round(new_score, 1),
                    "score_gain": round(gain, 1),
                    "action":     f"Khet ka {irr_val*100:.0f}% hissa seechain (irrigate karein).",
                    "why":        "Sahi irrigation ratio water stress kam karta hai aur yield badhata hai."
                })
        except Exception as e:
            print(f"     ⚠️  IRR={irr_val} scenario failed: {e}")

    suggestions.sort(key=lambda x: x["score_gain"], reverse=True)
    top = suggestions[:5]
    print(f"   └─ {len(top)} improvements found (out of {len(suggestions)} total)")

    # ── Step 4: Format result for LLM ──────────────────────
    def score_band(s):
        if s >= 65: return "Low Risk ✅ (Loan Eligible)"
        if s >= 45: return "Medium Risk ⚠️ (Small Loan)"
        return "High Risk ❌ (Loan Rejected)"

    if not top:
        return (
            f"📊 Current Status:\n"
            f"   Crop       : {base['Crop_Type']}\n"
            f"   Yield      : {current_yield:.0f} kg/ha\n"
            f"   Score      : {current_score:.1f}/100 — {score_band(current_score)}\n"
            f"   NPK        : {orig_npk:.0f} kg/ha\n"
            f"   Irrigation : {orig_irr:.0%}\n\n"
            "✅ Aapki farming already optimal hai! Koi bada improvement nahi mila.\n"
            "Suggestion: Local KVK (Krishi Vigyan Kendra) se crop-switch advice lein."
        )

    lines = [
        f"📊 Current Status:",
        f"   Crop       : {base['Crop_Type']}",
        f"   Yield      : {current_yield:.0f} kg/ha",
        f"   Score      : {current_score:.1f}/100 — {score_band(current_score)}",
        f"   NPK        : {orig_npk:.0f} kg/ha",
        f"   Irrigation : {orig_irr:.0%}",
        f"\n🔝 Top {len(top)} Improvements (XGBoost What-If Analysis):",
    ]
    for i, s in enumerate(top, 1):
        lines.append(
            f"\n  #{i} {s['change']}\n"
            f"      New Yield : {s['new_yield']:.0f} kg/ha\n"
            f"      New Score : {s['new_score']:.1f}/100 (+{s['score_gain']} points)\n"
            f"      New Band  : {score_band(s['new_score'])}\n"
            f"      Action    : {s['action']}\n"
            f"      Why       : {s['why']}"
        )

    lines.append(
        "\nINSTRUCTION FOR AI: Explain each suggestion in simple, warm Hinglish. "
        "Tell farmer EXACTLY what to do and what benefit they will get. "
        "Mention if they become loan eligible."
    )
    return "\n".join(lines)


# ══════════════════════════════════════════
#  STATE (add_messages = no loop bug)
# ══════════════════════════════════════════
class AgentState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]


# ══════════════════════════════════════════
#  MODEL — bind all 4 tools
# ══════════════════════════════════════════
model = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    api_key=os.getenv("GROK_API_KEY"),
).bind_tools([list_mandis, fetch_crop_price, get_weather, get_crop_advice])


# ══════════════════════════════════════════
#  GRAPH NODES — full debug output
# ══════════════════════════════════════════
async def call_model(state: AgentState) -> dict:
    _DBG["hop"] += 1
    hop = _DBG["hop"]

    # ── Debug: show full state entering agent ──
    print(f"\n{'='*55}")
    print(f"🤖 [HOP {hop}] AGENT NODE | messages in state: {len(state['messages'])}")
    for i, msg in enumerate(state["messages"]):
        t = type(msg).__name__
        c = str(msg.content)[:80].replace("\n", " ")
        tc = f" | tool_calls={[x['name'] for x in msg.tool_calls]}" \
             if hasattr(msg, "tool_calls") and msg.tool_calls else ""
        tn = f" | tool={msg.name}" if hasattr(msg, "name") and msg.name else ""
        print(f"   [{i}] {t}: '{c}'{tc}{tn}")
    print(f"{'='*55}")

    response = await model.ainvoke(
        [SystemMessage(content=SYSTEM_INSTRUCTIONS)] + state["messages"]
    )

    # ── Debug: LLM decision ──
    print(f"\n🧠 [HOP {hop}] LLM RESPONSE:")
    print(f"   content    : '{str(response.content)[:120]}'")
    print(f"   tool_calls : {response.tool_calls}")
# ══════════════════════════════════════════
#  GRAPH NODES — full debug output
# ══════════════════════════════════════════
async def call_model(state: AgentState) -> dict:
    _DBG["hop"] += 1
    hop = _DBG["hop"]

    print(f"\n{'='*55}")
    print(f"🤖 [HOP {hop}] AGENT | msgs in state: {len(state['messages'])}")
    for i, msg in enumerate(state["messages"]):
        t  = type(msg).__name__
        c  = str(msg.content)[:80].replace("\n"," ")
        tc = f" | calls={[x['name'] for x in msg.tool_calls]}" \
             if hasattr(msg,"tool_calls") and msg.tool_calls else ""
        tn = f" | tool={msg.name}" if hasattr(msg,"name") and msg.name else ""
        print(f"   [{i}] {t}: '{c}'{tc}{tn}")
    print(f"{'='*55}")

    response = await model.ainvoke(
        [SystemMessage(content=SYSTEM_INSTRUCTIONS)] + state["messages"]
    )

    # Debug LLM decision
    print(f"\n🧠 [HOP {hop}] LLM RESPONSE:")
    print(f"   content    : '{str(response.content)[:120]}'")
    print(f"   tool_calls : {response.tool_calls}")
    if response.tool_calls:
        for tc in response.tool_calls:
            name = tc["name"]
            _DBG["tools"][name] = _DBG["tools"].get(name, 0) + 1
            cnt = _DBG["tools"][name]
            flag = "  ⚠️ DUPLICATE CALL — LOOP RISK!" if cnt > 1 else ""
            print(f"   → Wants: {name}({tc['args']}){flag}")
    else:
        print(f"   → No tool call → going to END")

    return {"messages": [response]}


async def call_tool(state: AgentState) -> dict:
    last_msg = state["messages"][-1]
    results  = []

    print(f"\n{'─'*55}")
    print(f"🔧 TOOL NODE | executing {len(last_msg.tool_calls)} tool(s)")
    existing_tm = [m for m in state["messages"] if isinstance(m, ToolMessage)]
    print(f"   Existing ToolMessages in history: {len(existing_tm)}")

    for tool_call in last_msg.tool_calls:
        name = tool_call["name"]
        args = tool_call["args"]

        if name == "list_mandis":
            content = await list_mandis.ainvoke(args)
        elif name == "fetch_crop_price":
            content = await fetch_crop_price.ainvoke(args)
        elif name == "get_weather":
            content = await get_weather.ainvoke(args)
        elif name == "get_crop_advice":
            content = await get_crop_advice.ainvoke(args)
        else:
            content = f"❌ Unknown tool: {name}"
            print(f"   ⚠️  Unknown tool requested: {name}")

        results.append(
            ToolMessage(tool_call_id=tool_call["id"], content=str(content))
        )

    print(f"{'─'*55}")
    return {"messages": results}


def router(state: AgentState) -> str:
    last = state["messages"][-1]
    is_ai_with_tools = isinstance(last, AIMessage) and bool(last.tool_calls)

    print(f"\n🔀 ROUTER: last={type(last).__name__} | has_tool_calls={is_ai_with_tools}")
    if is_ai_with_tools:
        print(f"   → Route: tools")
        return "tools"
    print(f"   → Route: END")
    return END


# ══════════════════════════════════════════
#  BUILD GRAPH
# ══════════════════════════════════════════
graph = StateGraph(AgentState)
graph.add_node("agent", call_model)
graph.add_node("tools", call_tool)
graph.set_entry_point("agent")
graph.add_conditional_edges("agent", router, {"tools": "tools", END: END})
graph.add_edge("tools", "agent")
app = graph.compile()
print("✅ LangGraph compiled | Tools: list_mandis | fetch_crop_price | get_weather | get_crop_advice")


# ══════════════════════════════════════════
#  CHAT INTERFACE
# ══════════════════════════════════════════
async def chat_with_farmer():
    print(f"\n🌾 KisanSaathi | Farmer: {SESSION_FARMER_ID}")
    print("   Tools: list_mandis | fetch_crop_price | get_weather | get_crop_advice")
    print("   Type 'bye' to exit\n")

    state = {"messages": []}

    while True:
        try:
            user_input = input("👨‍🌾 Farmer: ").strip()
        except EOFError:
            break

        if not user_input:
            continue
        if user_input.lower() in ("quit","exit","bye","band karo"):
            print("🤖 KisanSaathi: Dhanyavad! Jai Jawan Jai Kisan 🌾")
            break

        # Reset debug counters for each new turn
        _DBG["hop"]   = 0
        _DBG["tools"] = {}
        print(f"\n{'#'*55}")
        print(f"# NEW TURN | Input: '{user_input}'")
        print(f"{'#'*55}")

        state["messages"].append(HumanMessage(content=user_input))

        try:
            result = await app.ainvoke(
                state,
                config={"recursion_limit": 10}
            )
            state["messages"] = result["messages"]
        except Exception as e:
            print(f"\n❌ GRAPH ERROR: {type(e).__name__}: {e}")
            print(f"   Hops completed: {_DBG['hop']}")
            print(f"   Tool call counts: {_DBG['tools']}")
            continue

        # Turn summary
        print(f"\n{'='*55}")
        print(f"✅ TURN COMPLETE | hops={_DBG['hop']} | tools={_DBG['tools']}")
        print(f"   Final msgs in state: {len(state['messages'])}")
        print(f"{'='*55}")

        # Print final AI response
        for msg in reversed(state["messages"]):
            if isinstance(msg, AIMessage) and not msg.tool_calls:
                print(f"\n🤖 KisanSaathi: {msg.content}\n")
                break


# Run in Jupyter:
await chat_with_farmer()

# Run as .py:
# if __name__ == "__main__":
#     asyncio.run(chat_with_farmer()) 


✅ XGBoost model + encoders loaded
✅ LangGraph compiled | Tools: list_mandis | fetch_crop_price | get_weather | get_crop_advice

🌾 KisanSaathi | Farmer: c59f6f44-1a98-4eaa-8cf0-3581316a32bb
   Tools: list_mandis | fetch_crop_price | get_weather | get_crop_advice
   Type 'bye' to exit


#######################################################
# NEW TURN | Input: 'Mein Loan ke liye Eligibe hu kii nahi Abhi meri Present score,yield,NPK intensitya nd IRR kya h aur mein inhe kitna Badhau??'
#######################################################

🤖 [HOP 1] AGENT | msgs in state: 1
   [0] HumanMessage: 'Mein Loan ke liye Eligibe hu kii nahi Abhi meri Present score,yield,NPK intensit'

🧠 [HOP 1] LLM RESPONSE:
   content    : ''
   tool_calls : [{'name': 'get_crop_advice', 'args': {}, 'id': 'cz3n1bh4k', 'type': 'tool_call'}]
   → Wants: get_crop_advice({})

🔀 ROUTER: last=AIMessage | has_tool_calls=True
   → Route: tools

───────────────────────────────────────────────────────
🔧 TOOL NODE | exec